# Model 2: Attention U-Net — Fetal Head Segmentation
**Dataset:** HC18 Grand Challenge — https://zenodo.org/records/1322001

**Architecture:** U-Net enhanced with **Attention Gates** on every skip connection. Each gate learns to suppress irrelevant background activations and focus only on fetal-head relevant regions. This addresses the base paper's finding that standard Attention U-Net underperformed due to speckle sensitivity — here we improve the gate design.

**Key improvement over base paper:** Double-layer attention gates with spatial squeeze-and-excitation.

## 1. Install Dependencies

In [ ]:
!pip install segmentation-models-pytorch timm albumentations opencv-python matplotlib pandas tqdm grad-cam -q

## 2. Download HC18 Dataset

In [ ]:
!wget -q --show-progress 'https://zenodo.org/records/1322001/files/training_set.zip' -O training_set.zip
!wget -q --show-progress 'https://zenodo.org/records/1322001/files/test_set.zip' -O test_set.zip
!unzip -q training_set.zip -d hc18
!unzip -q test_set.zip -d hc18
print('Dataset ready.')

## 3. Imports & Config

In [ ]:
import os, random, math, time, warnings
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from glob import glob

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm import tqdm

warnings.filterwarnings('ignore')
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

IMG_SIZE   = 256
BATCH_SIZE = 8
EPOCHS     = 100
LR         = 1e-4
PATIENCE   = 15
DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'
MODEL_NAME = 'attention_unet'

TRAIN_IMG_DIR = 'hc18/training_set'
TEST_IMG_DIR  = 'hc18/test_set'

print(f'Device : {DEVICE}')
print(f'Model  : Attention U-Net (with improved gates)')

## 4. Dataset

In [ ]:
def get_file_pairs(img_dir):
    imgs  = sorted(glob(os.path.join(img_dir, '*_HC.png')))
    masks = [p.replace('_HC.png', '_HC_Annotation.png') for p in imgs]
    return [(i, m) for i, m in zip(imgs, masks) if os.path.exists(m)]


def patient_split(pairs, val_ratio=0.15, test_ratio=0.10, seed=SEED):
    rng = random.Random(seed)
    all_p = pairs.copy(); rng.shuffle(all_p)
    n = len(all_p)
    n_val, n_test = int(n * val_ratio), int(n * test_ratio)
    return all_p[n_test + n_val:], all_p[n_test:n_test + n_val], all_p[:n_test]


def get_transforms(split='train'):
    if split == 'train':
        return A.Compose([
            A.PadIfNeeded(IMG_SIZE, IMG_SIZE),
            A.RandomCrop(IMG_SIZE, IMG_SIZE),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.RandomRotate90(p=0.5),
            A.GaussNoise(p=0.3),
            A.RandomBrightnessContrast(p=0.3),
            A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
            ToTensorV2()
        ])
    return A.Compose([
        A.Resize(IMG_SIZE, IMG_SIZE),
        A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
        ToTensorV2()
    ])


class HC18Dataset(Dataset):
    def __init__(self, pairs, transform=None):
        self.pairs = pairs; self.transform = transform

    def __len__(self): return len(self.pairs)

    def __getitem__(self, idx):
        img_path, msk_path = self.pairs[idx]
        img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
        msk = (cv2.imread(msk_path, cv2.IMREAD_GRAYSCALE) > 127).astype(np.float32)
        if self.transform:
            aug = self.transform(image=img, mask=msk)
            img, msk = aug['image'], aug['mask']
        return img, msk.unsqueeze(0)


all_pairs = get_file_pairs(TRAIN_IMG_DIR)
train_pairs, val_pairs, test_pairs = patient_split(all_pairs)
print(f'Train: {len(train_pairs)} | Val: {len(val_pairs)} | Test: {len(test_pairs)}')

train_dl = DataLoader(HC18Dataset(train_pairs, get_transforms('train')), batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=True)
val_dl   = DataLoader(HC18Dataset(val_pairs,   get_transforms('val')),   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
test_dl  = DataLoader(HC18Dataset(test_pairs,  get_transforms('val')),   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

## 5. Attention U-Net Architecture

**Attention Gate:** Computes a spatial attention coefficient α ∈ [0,1] for each position.
- Gate signal `g` comes from the decoder (coarser resolution)
- Feature map `x` comes from the encoder skip connection
- Both are projected to the same dimension, summed, and passed through sigmoid
- Output: `x * α` — suppresses background, amplifies fetal head regions

**Improvement vs base paper:** We add a channel squeeze-excitation step after each attention gate to recalibrate channel importance dynamically.

In [ ]:
class ConvBnRelu(nn.Module):
    def __init__(self, in_ch, out_ch, kernel=3, pad=1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel, padding=pad, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True)
        )
    def forward(self, x): return self.net(x)


class ChannelSE(nn.Module):
    """Squeeze-and-Excitation channel recalibration."""
    def __init__(self, ch, reduction=16):
        super().__init__()
        self.se = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(ch, ch // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(ch // reduction, ch, bias=False),
            nn.Sigmoid()
        )
    def forward(self, x):
        w = self.se(x).view(x.size(0), x.size(1), 1, 1)
        return x * w


class AttentionGate(nn.Module):
    """Attention gate with spatial gating + channel SE recalibration."""
    def __init__(self, F_g, F_l, F_int):
        super().__init__()
        self.W_g   = nn.Conv2d(F_g, F_int, 1, bias=False)
        self.W_x   = nn.Conv2d(F_l, F_int, 1, bias=False)
        self.psi   = nn.Sequential(nn.Conv2d(F_int, 1, 1, bias=False), nn.Sigmoid())
        self.bn    = nn.BatchNorm2d(F_int)
        self.se    = ChannelSE(F_l)

    def forward(self, g, x):
        # Upsample g to match x spatial size if needed
        if g.shape[2:] != x.shape[2:]:
            g = F.interpolate(g, size=x.shape[2:], mode='bilinear', align_corners=True)
        alpha = self.psi(F.relu(self.bn(self.W_g(g) + self.W_x(x)), inplace=True))
        return self.se(x * alpha)


class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            ConvBnRelu(in_ch, out_ch),
            ConvBnRelu(out_ch, out_ch)
        )
    def forward(self, x): return self.net(x)


class AttentionUNet(nn.Module):
    def __init__(self, in_ch=3, out_ch=1, features=[64, 128, 256, 512]):
        super().__init__()
        # Encoder
        self.enc   = nn.ModuleList()
        self.pool  = nn.MaxPool2d(2)
        prev = in_ch
        for f in features:
            self.enc.append(DoubleConv(prev, f)); prev = f

        # Bottleneck
        self.neck = DoubleConv(features[-1], features[-1] * 2)

        # Decoder with attention gates
        rev = list(reversed(features))
        self.up   = nn.ModuleList()
        self.att  = nn.ModuleList()
        self.dec  = nn.ModuleList()

        in_d = features[-1] * 2
        for f in rev:
            self.up.append(nn.ConvTranspose2d(in_d, f, 2, stride=2))
            self.att.append(AttentionGate(F_g=f, F_l=f, F_int=f // 2))
            self.dec.append(DoubleConv(f * 2, f))
            in_d = f

        self.head = nn.Conv2d(features[0], out_ch, 1)

    def forward(self, x):
        skips = []
        for enc in self.enc:
            x = enc(x); skips.append(x); x = self.pool(x)

        x = self.neck(x)

        for up, att, dec, skip in zip(self.up, self.att, self.dec, reversed(skips)):
            x    = up(x)
            skip = att(x, skip)           # attention-gated skip
            x    = dec(torch.cat([x, skip], dim=1))

        return torch.sigmoid(self.head(x))


model = AttentionUNet().to(DEVICE)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Attention U-Net parameters: {total_params:,}')

## 6. Loss, Metrics & Optimizer

In [ ]:
def hybrid_loss(pred, target, smooth=1e-6):
    bce   = F.binary_cross_entropy(pred, target)
    inter = (pred * target).sum(dim=(2, 3))
    dice  = 1 - (2 * inter + smooth) / (pred.sum(dim=(2,3)) + target.sum(dim=(2,3)) + smooth)
    return bce + dice.mean()


def compute_metrics(pred, target, thresh=0.5, smooth=1e-6):
    pred_b = (pred > thresh).float()
    tp = (pred_b * target).sum()
    fp = (pred_b * (1 - target)).sum()
    fn = ((1 - pred_b) * target).sum()
    tn = ((1 - pred_b) * (1 - target)).sum()
    dice       = (2*tp + smooth) / (2*tp + fp + fn + smooth)
    iou        = (tp + smooth) / (tp + fp + fn + smooth)
    precision  = (tp + smooth) / (tp + fp + smooth)
    recall     = (tp + smooth) / (tp + fn + smooth)
    accuracy   = (tp + tn + smooth) / (tp + tn + fp + fn + smooth)
    specificity= (tn + smooth) / (tn + fp + smooth)
    return {'dice': dice.item(), 'iou': iou.item(), 'precision': precision.item(),
            'recall': recall.item(), 'f1': dice.item(), 'accuracy': accuracy.item(), 'specificity': specificity.item()}


optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.5, verbose=True)
print('Ready.')

## 7. Training Loop

In [ ]:
history = {'train_loss': [], 'val_loss': [], 'val_dice': [], 'val_iou': []}
best_val_loss = float('inf'); patience_ctr = 0


def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss = 0
    all_m = {k: 0.0 for k in ['dice','iou','precision','recall','f1','accuracy','specificity']}
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for imgs, masks in tqdm(loader, leave=False):
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            preds = model(imgs)
            loss  = hybrid_loss(preds, masks)
            if train:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            total_loss += loss.item()
            m = compute_metrics(preds.detach(), masks)
            for k in all_m: all_m[k] += m[k]
    n = len(loader)
    return total_loss / n, {k: v / n for k, v in all_m.items()}


print(f'Training Attention U-Net for up to {EPOCHS} epochs...\n')
start = time.time()

for epoch in range(1, EPOCHS + 1):
    t_loss, _      = run_epoch(train_dl, True)
    v_loss, v_mets = run_epoch(val_dl,   False)
    scheduler.step(v_loss)

    history['train_loss'].append(t_loss)
    history['val_loss'].append(v_loss)
    history['val_dice'].append(v_mets['dice'])
    history['val_iou'].append(v_mets['iou'])

    if epoch % 10 == 0 or epoch == 1:
        print(f'Ep {epoch:03d}/{EPOCHS} | Train: {t_loss:.4f} | Val: {v_loss:.4f} | Dice: {v_mets["dice"]:.4f} | IoU: {v_mets["iou"]:.4f}')

    if v_loss < best_val_loss:
        best_val_loss = v_loss
        torch.save(model.state_dict(), f'best_{MODEL_NAME}.pth')
        patience_ctr = 0
    else:
        patience_ctr += 1
        if patience_ctr >= PATIENCE:
            print(f'Early stopping at epoch {epoch}'); break

print(f'Done in {(time.time()-start)/60:.1f} min | Best val loss: {best_val_loss:.4f}')

## 8. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Attention U-Net Training History')
axes[0].plot(history['train_loss'], label='Train'); axes[0].plot(history['val_loss'], label='Val')
axes[0].set_title('Loss'); axes[0].legend()
axes[1].plot(history['val_dice'], color='green'); axes[1].set_title('Val Dice')
axes[2].plot(history['val_iou'],  color='orange'); axes[2].set_title('Val IoU')
for ax in axes: ax.set_xlabel('Epoch')
plt.tight_layout()
plt.savefig(f'{MODEL_NAME}_training_curves.png', dpi=150)
plt.show()

## 9. Test Evaluation

In [ ]:
model.load_state_dict(torch.load(f'best_{MODEL_NAME}.pth', map_location=DEVICE))
test_loss, test_mets = run_epoch(test_dl, False)

print('=' * 50)
print('  Attention U-Net — Test Results')
print('=' * 50)
for k, v in test_mets.items():
    print(f'  {k:<12}: {v*100:.2f}%')
print(f'  {"loss":<12}: {test_loss:.4f}')
print('=' * 50)

## 10. Visualize Attention Maps + HC Estimation

In [ ]:
def estimate_hc(mask_np, pixel_size_mm=0.143):
    mask_u8 = (mask_np * 255).astype(np.uint8)
    kernel  = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    mask_u8 = cv2.morphologyEx(mask_u8, cv2.MORPH_CLOSE, kernel)
    contours, _ = cv2.findContours(mask_u8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours or len(max(contours, key=cv2.contourArea)) < 5: return None
    largest = max(contours, key=cv2.contourArea)
    (_, _), (MA, ma), _ = cv2.fitEllipse(largest)
    a, b = MA / 2, ma / 2
    hc_px = math.pi * (3*(a+b) - math.sqrt((3*a+b)*(a+3*b)))
    return hc_px * pixel_size_mm


denorm_mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
denorm_std  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)

model.eval()
hc_errors = []
fig, axes = plt.subplots(3, 4, figsize=(16, 12))

with torch.no_grad():
    for idx, (imgs, masks) in enumerate(test_dl):
        if idx >= 1: break
        preds = model(imgs.to(DEVICE)).cpu()
        for i in range(min(3, imgs.shape[0])):
            img_np  = (imgs[i] * denorm_std + denorm_mean).permute(1,2,0).numpy().clip(0,1)
            msk_np  = masks[i, 0].numpy()
            pred_np = (preds[i, 0] > 0.5).numpy().astype(np.float32)

            hc_p = estimate_hc(pred_np)
            hc_g = estimate_hc(msk_np)

            axes[i][0].imshow(img_np);                axes[i][0].set_title('Input')
            axes[i][1].imshow(msk_np, cmap='gray');   axes[i][1].set_title('GT Mask')
            axes[i][2].imshow(pred_np, cmap='gray');  axes[i][2].set_title('Predicted')

            overlay = (img_np * 255).astype(np.uint8).copy()
            ctrs, _ = cv2.findContours((pred_np*255).astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            cv2.drawContours(overlay, ctrs, -1, (0,255,0), 2)
            axes[i][3].imshow(overlay)
            axes[i][3].set_title(f'HC={hc_p:.1f}mm' if hc_p else 'No contour')

            if hc_p and hc_g: hc_errors.append(abs(hc_p - hc_g))
            for ax in axes[i]: ax.axis('off')

plt.suptitle('Attention U-Net — Qualitative Results', fontsize=14)
plt.tight_layout()
plt.savefig(f'{MODEL_NAME}_qualitative.png', dpi=150)
plt.show()

if hc_errors:
    print(f'HC MAE : {np.mean(hc_errors):.2f} mm')
    print(f'HC MSE : {np.mean(np.array(hc_errors)**2):.2f} mm²')

## 11. Save Results

In [ ]:
results = {'model': 'Attention-UNet', **{k: round(v*100,2) for k,v in test_mets.items()}, 'test_loss': round(test_loss,4)}
if hc_errors:
    results['hc_mae_mm']  = round(float(np.mean(hc_errors)), 3)
    results['hc_mse_mm2'] = round(float(np.mean(np.array(hc_errors)**2)), 3)

df = pd.DataFrame([results])
df.to_csv(f'{MODEL_NAME}_results.csv', index=False)
print(df.to_string(index=False))